In [3]:
# ⚙️ Global Config & Services (using centralized modules)

import json
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# Add parent directory to path and change to project root
import os

# Get the notebook's current directory and find project root
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Change to project root and add to path
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📂 Working directory: {os.getcwd()}")

from src.services.llm_services import (
    load_config,
    get_llm,
    get_text_embeddings,
    validate_api_keys,
    print_config_summary
)

# Load environment variables
load_dotenv()

# Load configuration from config.yaml (now we're in project root)
config = load_config("src/config/config.yaml")

# Validate API keys
validate_api_keys(config, verbose=True)

# Print summary
print_config_summary(config)


📂 Working directory: c:\Users\User\Desktop\Operation Legder-Mind
✅ Config loaded:
  LLM: openrouter (openai/gpt-4o-mini)
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:379: UserWarning: ⚠️  OPENAI_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:379: UserWarning: ⚠️  GROQ_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:379: UserWarning: ⚠️  COHERE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")


In [4]:
# Initialize LLM and Embeddings using factories from llm_services
llm = get_llm(config)
embeddings = get_text_embeddings(config)

print(f"✅ LLM: {config['llm_provider']} / {config['llm_model']}")
print(f"✅ Embeddings: {config['text_emb_provider']} / {config['text_emb_model']}")

# Verify API key with test completion
print("\n🔍 Testing LLM API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"✅ LLM API verified: {test_msg[:50]}")
except Exception as e:
    print(f"❌ LLM API test failed: {e}")
    print("⚠️  Please check your .env file and API key configuration.")


c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:133: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ LLM: openrouter / gpt-4o-mini
✅ Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2

🔍 Testing LLM API connection...
❌ LLM API test failed: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402, 'metadata': {'limit_source': 'openrouter_credits', 'remedy_hint': 'Add credits at https://openrouter.ai/settings/credits, or lower max_tokens / prompt size to fit your remaining balance.'}}}
⚠️  Please check your .env file and API key configuration.


**Document Loading**

In [5]:
from langchain_community.document_loaders import (
                                                PyPDFLoader, 
                                                DirectoryLoader, 
                                                TextLoader
                                                )

from langchain_core.documents import Document

pdf_dir = Path(config["data_root"]) / "pdfs"
pdf_dir.mkdir(parents=True, exist_ok=True)


# Try loading PDFs
pdf_files = list(pdf_dir.glob("*.pdf"))
documents=[]
if len(pdf_files) == 0:
    print("⚠️ No PDFs found. Creating sample document...")

    sample_content = """
    Annual Financial Report - 2025

    Company Overview:
    ABC Technologies is a financial technology company
    providing digital payment and banking solutions.

    Revenue:
    During the financial year 2025, the company generated
    total revenue of $5,000,000. Subscription services
    contributed $2,000,000, while transaction fees
    generated $3,000,000.

    Operating Expenses:
    The company reported operating expenses of $3,200,000.
    Employee salaries accounted for $1,500,000.
    Infrastructure costs were $800,000.
    Marketing expenses were $500,000.
    Administrative expenses were $400,000.

    Net Profit:
    The company's net profit for 2025 was $1,800,000.

    Financial Performance:
    Revenue increased by 15% compared to the previous
    financial year. Operating expenses increased by 8%.

    Cash Flow:
    Operating activities generated $2,100,000 in cash.
    Investing activities consumed $600,000.
    Financing activities generated $200,000.

    Risk Management:
    The company identified foreign exchange fluctuations,
    cybersecurity threats, and regulatory changes as
    significant business risks.
    """

    documents.append(
        Document(
            page_content=sample_content,
            metadata={
                "source": "sample_financial_report.txt",
                "page": 1,
                "document_type": "sample"
            }
        )
    )
else:
    for pdf_path in pdf_files:
        loader = PyPDFLoader(str(pdf_path))
        docs = loader.load()
        documents.extend(docs)



print(f"✅ Loaded {len(documents)} document pages")
print(f"  Total characters: {sum(len(d.page_content) for d in documents):,}")



✅ Loaded 142 document pages
  Total characters: 639,429


# Clean Document 

In [6]:
import re

def clean_document(text: str) -> str:

    # Remove page numbers such as Page 25
    text = re.sub(
        r"(?im)^\s*page\s+\d+\s*$",
        "",
        text
    )

    # Remove extra whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


for doc in documents:
    doc.page_content = clean_document(doc.page_content)

---

#  Chunking 


In [7]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)

def get_splitter(strategy: str, chunk_size: int = 1500, chunk_overlap: int = 150):
    """
    Return a text splitter based on the specified strategy.
    
    Args:
        strategy: One of "recursive", "fixed", or "sentence"
        chunk_size: Maximum characters per chunk
        chunk_overlap: Overlap between consecutive chunks
        
    Returns:
        A LangChain text splitter instance
    """
    if strategy == "recursive": 
        return RecursiveCharacterTextSplitter(
                                            chunk_size=chunk_size,
                                            chunk_overlap=chunk_overlap,
                                            separators=["\n\n", "\n", ". ", ", ", " ", ""]
                                            )

    elif strategy == "fixed":
        return CharacterTextSplitter(
                                    chunk_size=chunk_size,
                                    chunk_overlap=chunk_overlap,
                                    separator=" "
                                    )

    elif strategy == "sentence":
        pass 

    else:
        raise ValueError("Unknown stratergy")


# Test all strategies
# strategies = ["recursive", "fixed", "sentence"]
strategies = ["recursive"]
split_results = {}

for strategy in strategies:
    splitter = get_splitter(strategy)
    chunks = splitter.split_documents(documents)
    
    # Add strategy to metadata
    for chunk in chunks: 
        chunk.metadata["splitter"] = strategy
    
    split_results[strategy] = chunks
    print(f"✅ {strategy:10s}: {len(chunks):4d} chunks")

if split_results["recursive"]:
    example = split_results["recursive"][0]

    print(f"Length: {len(example.page_content)} chars")
    print(f"Content: {example.page_content[:150]}...")
    print(f"Metadata: {example.metadata}")
else:
    print("No chunks generated.")


✅ recursive :  543 chunks
Length: 29 chars
Content: On Our Way
2024 ANNUAL REPORT...
Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2025-03-20T10:48:37+05:30', 'moddate': '2025-03-22T09:33:29-04:00', 'trapped': '/False', 'source': 'data\\pdfs\\AEE Mini Project 01 2024 Annual Report.pdf', 'total_pages': 142, 'page': 0, 'page_label': 'a', 'splitter': 'recursive'}


## Generate Questions

In [ ]:
llm_a = get_llm({**config, "temperature": 0.3})
llm_b = get_llm({**config, "temperature": 0.0})

# ----------------------------------------------------------------------------
# 2. Prompts & Pipeline Functions for Step A & Step B
# ----------------------------------------------------------------------------
import re
import json
import time
from typing import List, Dict, Any
from langchain_core.prompts import ChatPromptTemplate

def extract_json_from_response(text: str) -> Any:
    """Extract JSON payload from LLM responses, stripping code fences."""
    text = text.strip()
    code_block = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if code_block:
        text = code_block.group(1).strip()
    else:
        array_match = re.search(r"(\[[\s\S]*\])", text)
        if array_match:
            text = array_match.group(1).strip()
    return json.loads(text)

# Step A Prompt: 10 Questions strictly grounded in chunk
prompt_template_a = ChatPromptTemplate.from_template(
    """You are an expert Q&A generator (LLM A) for enterprise financial reports.

Your task is to generate exactly 10 high-quality, diverse questions based STRICTLY on the text chunk provided below.

## Question Categories

Generate questions across these three categories:

1. Hard Facts (4 questions)
   - Quantitative and financial metrics (figures, percentages, dates).
   - Factual and historical details (events, corporate milestones).
   - Governance and risk information explicitly mentioned in the context.

2. Strategic Summaries (3 questions)
   - Business strategies and operational details.
   - Products, business segments, and performance drivers.
   - Summaries of important business developments and their relationships.

3. Stylistic / Creative (3 questions)
   - Ask for explanations of information in simple language.
   - Request executive-style summaries of information in the chunk.
   - Ask for information to be presented in a different format, such as bullet points or a short briefing.

## Rules

1. Every question must be answerable using ONLY information in the provided chunk.
2. Do NOT assume, extrapolate, or introduce external facts.
3. Do NOT generate questions about information absent from the chunk.
4. Avoid duplicate or repetitive questions.
5. Questions must be clear, concise, and self-contained.
6. For stylistic questions, change only the presentation style, never the underlying facts.
7. If the chunk does not contain enough information for a category, do not invent questions. Generate other answerable questions from the available content and indicate their actual category.

## Output Format

Return ONLY a valid JSON array containing exactly 10 objects:

[
  {
    "category": "hard_facts",
    "question": "Question 1"
  },
  {
    "category": "strategic_summaries",
    "question": "Question 5"
  },
  {
    "category": "stylistic_creative",
    "question": "Question 8"
  }
]

Text Chunk:
{context}
"""
)

def generate_questions_step_a(chunk_text: str, llm: llm_a, retries: int = 3) -> List[str]:
    chain_a = prompt_template_a | llm
    for attempt in range(retries):
        try:
            response = chain_a.invoke({"context": chunk_text})
            questions = extract_json_from_response(response.content)
            if isinstance(questions, list) and len(questions) > 0:
                return [str(q).strip() for q in questions[:10]]
        except Exception as e:
            if attempt == retries - 1:
                raise RuntimeError(f"Step A failed: {e}")
            time.sleep(1.5 * (attempt + 1))
    return []

# Step B Prompt: Answer 10 questions using context + questions
prompt_template_b = ChatPromptTemplate.from_template(
    """You are an authoritative financial analyst and answering system (LLM B).

Your task is to generate accurate, comprehensive answers for all questions using STRICTLY the provided context chunk.

## Rules

1. Answer every question using ONLY the provided context.
2. Be precise, factual, and complete.
3. Include relevant figures, metrics, and dates when mentioned.
4. Do NOT hallucinate, infer unsupported conclusions, or introduce outside knowledge.
5. If the context does not contain sufficient information to answer a question, respond exactly:
   "Information not provided in the context."
6. Preserve the original question text and its category.
7. For stylistic/creative questions, change the presentation style as requested while preserving factual accuracy.
8. Do not generate additional questions or answers.

## Output Format

Return ONLY a valid JSON array of objects:

[
  {
    "category": "hard_facts",
    "question": "Original question",
    "answer": "Answer based on the context"
  },
  {
    "category": "strategic_summaries",
    "question": "Original question",
    "answer": "Answer based on the context"
  },
  {
    "category": "stylistic_creative",
    "question": "Original question",
    "answer": "Answer based on the context"
  }
]

Context Chunk:
{context}

Questions:
{questions}
"""
)

def generate_answers_step_b(
    chunk_text: str, questions: List[str], llm: llm_b, retries: int = 3
) -> List[Dict[str, str]]:
    chain_b = prompt_template_b | llm
    questions_json = json.dumps(questions, indent=2)
    for attempt in range(retries):
        try:
            response = chain_b.invoke({"context": chunk_text, "questions": questions_json})
            qa_pairs = extract_json_from_response(response.content)
            if isinstance(qa_pairs, list) and len(qa_pairs) > 0:
                return [
                    {"question": str(item["question"]).strip(), "answer": str(item["answer"]).strip()}
                    for item in qa_pairs if isinstance(item, dict) and "question" in item and "answer" in item
                ]
        except Exception as e:
            if attempt == retries - 1:
                raise RuntimeError(f"Step B failed: {e}")
            time.sleep(1.5 * (attempt + 1))
    return []

In [ ]:
chunks = split_results.get("recursive", [])
assert len(chunks) > 0, "No chunks found in split_results['recursive']"
test_chunk = chunks[0]
print(f"📄 Testing on Chunk 1 ({len(test_chunk.page_content)} characters):\n")
# Step A: 10 Questions
print("⏳ Step A: LLM A generating 10 questions...")
test_questions = generate_questions_step_a(test_chunk.page_content, llm_a)
for i, q in enumerate(test_questions, 1):
    print(f"  [{i:02d}] {q}")
# Step B: 10 Answers
print("\n⏳ Step B: LLM B generating 10 answers...")
test_qa_pairs = generate_answers_step_b(test_chunk.page_content, test_questions, llm_b)
for i, pair in enumerate(test_qa_pairs, 1):
    print(f"  Q{i:02d}: {pair['question']}")
    print(f"  A{i:02d}: {pair['answer']}\n")

📄 Testing on Chunk 1 (29 characters):

⏳ Step A: LLM A generating 10 questions...
  [01] What is the title of the document referenced in the text?
  [02] What year does the annual report pertain to?
  [03] Is the annual report for the year 2024 or 2023?
  [04] What type of document is mentioned in the text?
  [05] Does the text indicate that the report is for a specific company?
  [06] What is the primary focus of the document?
  [07] Is there any mention of financial metrics in the text?
  [08] Are there any operational details provided in the text?
  [09] Does the text reference any corporate milestones?
  [10] Is there any information about governance or risk factors in the text?

⏳ Step B: LLM B generating 10 answers...
  Q01: What is the title of the document referenced in the text?
  A01: On Our Way

  Q02: What year does the annual report pertain to?
  A02: 2024

  Q03: Is the annual report for the year 2024 or 2023?
  A03: 2024

  Q04: What type of document is mentioned in the 

In [ ]:
# ----------------------------------------------------------------------------
# 4. Full Generation Loop with Checkpointing (JSONL)
# ----------------------------------------------------------------------------
from tqdm.auto import tqdm
from pathlib import Path

output_dir = Path(config.get("artifacts_root", "./artifacts")) / "data"
output_dir.mkdir(parents=True, exist_ok=True)
jsonl_file = output_dir / "qa_dataset.jsonl"

processed_chunk_ids = set()
if jsonl_file.exists():
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    record = json.loads(line)
                    if "chunk_id" in record:
                        processed_chunk_ids.add(record["chunk_id"])
                except json.JSONDecodeError:
                    pass

print(f"🔄 Already processed chunks: {len(processed_chunk_ids)} / {len(chunks)}")

# Set MAX_CHUNKS to None to run all 543 chunks, or e.g. 10 for quick testing
MAX_CHUNKS = 50
RATE_LIMIT_DELAY = 0.5

chunks_to_process = chunks if MAX_CHUNKS is None else chunks[:MAX_CHUNKS]
total_qa_generated = 0
failed_chunks = []

with open(jsonl_file, "a", encoding="utf-8") as out_f:
    pbar = tqdm(enumerate(chunks_to_process), total=len(chunks_to_process), desc="Q/A Factory")
    for chunk_idx, chunk in pbar:
        if chunk_idx in processed_chunk_ids:
            continue
        chunk_text = chunk.page_content.strip()
        if len(chunk_text) < 150:
            continue
        try:
            questions = generate_questions_step_a(chunk_text, llm_a)
            qa_pairs = generate_answers_step_b(chunk_text, questions, llm_b)
            for q_idx, pair in enumerate(qa_pairs, 1):
                record = {
                    "qa_id": f"chunk_{chunk_idx:04d}_q{q_idx:02d}",
                    "chunk_id": chunk_idx,
                    "question": pair["question"],
                    "answer": pair["answer"],
                    "context": chunk_text,
                    "metadata": {
                        "source": chunk.metadata.get("source", "unknown"),
                        "page": chunk.metadata.get("page", 0),
                        "splitter": chunk.metadata.get("splitter", "recursive"),
                    }
                }
                out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                total_qa_generated += 1
            out_f.flush()
            processed_chunk_ids.add(chunk_idx)
            pbar.set_postfix({"chunk": chunk_idx, "total_qa": total_qa_generated})
            if RATE_LIMIT_DELAY > 0:
                time.sleep(RATE_LIMIT_DELAY)
        except Exception as e:
            failed_chunks.append((chunk_idx, str(e)))

print(f"🎉 Generation Complete! Generated {total_qa_generated} Q/A pairs.")

🔄 Already processed chunks: 43 / 543


Q/A Factory:   0%|          | 0/50 [00:00<?, ?it/s]

🎉 Generation Complete! Generated 20 Q/A pairs.


In [ ]:
# ----------------------------------------------------------------------------
# 5. Dataset Inspection & Multi-Format Export (JSON, CSV, Parquet)
# ----------------------------------------------------------------------------
import pandas as pd
import json
from pathlib import Path

output_dir = Path(config.get("artifacts_root", "./artifacts")) / "data"
jsonl_file = output_dir / "qa_dataset.jsonl"

records = []
if jsonl_file.exists():
    with open(jsonl_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

df_qa = pd.DataFrame(records)
print(f"📊 Dataset Summary:")
print(f"  Total Q/A Pairs:     {len(df_qa):,}")
print(f"  Unique Chunks:       {df_qa['chunk_id'].nunique() if not df_qa.empty else 0:,}")

if not df_qa.empty:
    df_qa["page"] = df_qa["metadata"].apply(lambda m: m.get("page", 0) if isinstance(m, dict) else 0)
    df_qa["source"] = df_qa["metadata"].apply(lambda m: m.get("source", "") if isinstance(m, dict) else "")
    
    json_path = output_dir / "qa_dataset.json"
    csv_path = output_dir / "qa_dataset.csv"
    parquet_path = output_dir / "qa_dataset.parquet"
    
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
        
    df_qa.to_csv(csv_path, index=False, encoding="utf-8")
    try:
        df_qa.to_parquet(parquet_path, index=False)
        print(f"  💾 Parquet saved:    {parquet_path}")
    except Exception:
        pass
        
    print(f"  💾 JSONL saved:      {jsonl_file}")
    print(f"  💾 JSON saved:       {json_path}")
    print(f"  💾 CSV saved:        {csv_path}")
    
    print("\n🔍 Sample Generated Q/A Pairs:")
    display(df_qa[["qa_id", "question", "answer", "page"]].head(5))

# Split the data

In [ ]:
# ----------------------------------------------------------------------------
# 6. Train / Test Split (80% Train, 20% Test — Chunk-Aware)
# ----------------------------------------------------------------------------
import json
import random
from pathlib import Path
from collections import defaultdict

data_dir = Path(config.get("artifacts_root", "./artifacts")) / "data"
input_file = data_dir / "qa_dataset.jsonl"
train_file = data_dir / "train.jsonl"
test_file = data_dir / "test.jsonl"

# 1. Load all records
records = []
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"📦 Total Q/A pairs loaded: {len(records)}")

# 2. Group records by chunk_id to prevent data leakage
chunks_map = defaultdict(list)
for r in records:
    chunks_map[r["chunk_id"]].append(r)

unique_chunk_ids = list(chunks_map.keys())
random.seed(42)  # For reproducible splits
random.shuffle(unique_chunk_ids)

# 3. 80% train, 20% test on chunk groups
split_idx = int(len(unique_chunk_ids) * 0.8)
train_chunk_ids = set(unique_chunk_ids[:split_idx])
test_chunk_ids = set(unique_chunk_ids[split_idx:])

train_records = [r for cid in train_chunk_ids for r in chunks_map[cid]]
test_records = [r for cid in test_chunk_ids for r in chunks_map[cid]]

# 4. Shuffle pairs within each split
random.shuffle(train_records)
random.shuffle(test_records)

# 5. Save to disk
with open(train_file, "w", encoding="utf-8") as f:
    for r in train_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(test_file, "w", encoding="utf-8") as f:
    for r in test_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\n✅ Split Complete (80% Train / 20% Test):")
print(f"  Train: {len(train_records):4d} pairs ({len(train_chunk_ids):2d} chunks) ➔ {train_file}")
print(f"  Test:  {len(test_records):4d} pairs ({len(test_chunk_ids):2d} chunks) ➔ {test_file}")

📦 Total Q/A pairs loaded: 450

✅ Split Complete (80% Train / 20% Test):
  Train:  360 pairs (36 chunks) ➔ artifacts\data\train.jsonl
  Test:    90 pairs ( 9 chunks) ➔ artifacts\data\test.jsonl


## Add the dataset to hf

In [ ]:
from datasets import load_dataset

# Your local dataset paths
data_files = {
    "train": "../artifacts/data/train.jsonl",
    "test": "../artifacts/data/test.jsonl"
}

# Load JSONL files
dataset = load_dataset(
    "json",
    data_files=data_files
)

print(dataset)
print(dataset["train"][0])

# Push to Hugging Face
dataset.push_to_hub(
    "janadaj/operation-ledger-mind-qa",
    private=True
)

print("Dataset uploaded successfully!")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['qa_id', 'chunk_id', 'question', 'answer', 'context', 'metadata'],
        num_rows: 360
    })
    test: Dataset({
        features: ['qa_id', 'chunk_id', 'question', 'answer', 'context', 'metadata'],
        num_rows: 90
    })
})
{'qa_id': 'chunk_0181_q06', 'chunk_id': 181, 'question': 'What type of licenses is the company seeking to optimize its payment solutions?', 'answer': 'The company is seeking payment or other financial licenses.', 'context': 'several other jurisdictions to optimize our payment solutions and support the future growth of our business. We could be denied  such \nlicenses, have existing licenses revoked, or be required to make significant changes to our busin ess operations before being gr anted \nsuch licenses. If we are denied payment or other financial licen ses or such licenses are revoked, we could be forced to cease o r limit', 'metadata': {'source': 'data\\pdfs\\AEE Mini Project 01 2024 Annual Report.p

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

... (6 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Dataset uploaded successfully!


# Add to weaviate (Embedding)

In [ ]:
import weaviate
import os
from langchain_weaviate import WeaviateVectorStore

# 1. Collect chunks from split_results
chunks_to_index = split_results.get("recursive", [])
if not chunks_to_index:
    chunks_to_index = [chunk for chunks in split_results.values() for chunk in chunks]
print(f"📦 Total chunks to embed: {len(chunks_to_index)}")

# 2. Connect to Weaviate Client
weaviate_url = os.getenv("WEAVIATE_URL")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")

if weaviate_url and weaviate_api_key:
    print(f"🌐 Connecting to Weaviate Cloud: {weaviate_url}...")
    client = weaviate.connect_to_weaviate_cloud(
        cluster_url=weaviate_url,
        auth_credentials=weaviate.auth.AuthApiKey(weaviate_api_key)
    )
else:
    # Embedded Weaviate (Works automatically in Google Colab & Linux)
    try:
        print("💻 Connecting to Embedded Weaviate (in-process)...")
        client = weaviate.connect_to_embedded()
    except Exception:
        print("🐳 Falling back to local Docker Weaviate (localhost:8080)...")
        client = weaviate.connect_to_local()


# 3. Create or replace the collection & embed documents
INDEX_NAME = "FinancialReportChunks"

# Delete previous collection if re-indexing to avoid duplicate entries
if client.collections.exists(INDEX_NAME):
    print(f"🔄 Removing existing collection '{INDEX_NAME}' for clean re-indexing...")
    client.collections.delete(INDEX_NAME)
print(f"🚀 Embedding and indexing {len(chunks_to_index)} chunks with {config['text_emb_provider']}...")
vectorstore = WeaviateVectorStore.from_documents(
    documents=chunks_to_index,
    embedding=embeddings,    
    client=client,
    index_name=INDEX_NAME
)
print(f"✅ Successfully indexed {len(chunks_to_index)} chunks in Weaviate collection '{INDEX_NAME}'!")



WeaviateStartUpError: Windows is not supported with EmbeddedDB. Please upvote this feature request if you want
                 this: https://github.com/weaviate/weaviate/issues/3315